# LeNet-1 (1989) Implementation

This notebook is reimplementation of the original Convnet paper from LeCun et.al used for digit recognition in handwritten zip codes [^1]. The neural net was then evolved from this first version over a decade [^2].

## References
[^1]: LeCun, Y. et al. (1989). Backpropagation applied to Handwritten Zip Code Recognition. [Link to PDF](http://yann.lecun.com/exdb/publis/pdf/lecun-89e.pdf)

[^2]: LeCun, Y. et al. (1998) GradientBased Learning Applied to Document Recognition



In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)

## Table of Contents
1. [Data Preparation](#data-preparation)
2. [Model Architecture](#model-architecture)
3. [Forward Pass](#forward-pass)
4. [Training Loop](#training-loop)
5. [Results](#results)

## Data Preparation

The data pre-processing is done in the notebook **pytorch_playground**, which load, resizes to 16x16 image from 28x28 image in MNIST Dataset, and stores the subset of the dataset to npz file in data directory.
* The saved dataset contains 7291 training and 2007 test images of handwritten digits and corresponding labels (0-9)

In [7]:
data_train = np.load("./data/train1989.npz")
data_test = np.load("./data/test1989.npz")

# Access arrays inside
Xtr = torch.from_numpy(data_train["X"])
Ytr = torch.from_numpy(data_train["Y"])

Xte = torch.from_numpy(data_test["X"])
Yte = torch.from_numpy(data_test["Y"])

**Task 4** Check the shape of the training and test data and labels.

In [8]:
print(f"Training data shape: {Xtr.shape}")
print(f"Training labels shape: {Ytr.shape}")
print(f"Test data shape: {Xte.shape}")
print(f"Test labels shape: {Yte.shape}")

Training data shape: torch.Size([7291, 1, 16, 16])
Training labels shape: torch.Size([7291, 10])
Test data shape: torch.Size([2007, 1, 16, 16])
Test labels shape: torch.Size([2007, 10])


## Model Architecture

Score function is the mathematical mapping that takes the input (e.g. image of handwritten digit) and transforms into scores for each possible class.  In other words, forward pass of a neural network is the score function. The simplest being $$f(x, W, b) = f_a(\sum_{i=1}^{n} w_i x_i + b)$$
* $W$: (weights) are the parameters of the neural network
* $x$: features chosen for the problem (in this case, grayscale value of pixels)
* $f_a$: chosen activation function for introducing non-linearity (e.g. tanh or relu)

### Score function: Weights and bias
The notebook uses paper implementation of the weight initialization which requires fan_in(number of input connections to a neuron) which prevents vanishing gradients.

In [9]:
def init_weights(fan_in, *shape):
    """Weight initialization as described in the paper"""
    weight = (torch.rand(*shape) - 0.5) * 2 * 2.4 / fan_in
    return nn.Parameter(weight)

**Task**:

In [10]:
macs = 0  # keep track of MACs (multiply accumulates)
acts = 0  # keep track of number of activations

# H1 layer parameters
H1w = init_weights(5*5*1, 12, 1, 5, 5)
H1b = torch.zeros(12, 8, 8)
assert H1w.numel() + H1b.numel() == 1068
macs = 5*5*8*8*12 + 768
acts += (8*8) * 12
assert macs == 19968

# H2 layer parameters
H2w = init_weights(5*5*8, 12, 8, 5, 5)
H2b = torch.zeros(12, 4, 4)
assert H2w.numel() + H2b.numel() == 2592
macs += (5*5*8) * (4*4) * 12 + 192
acts += (4*4) * 12
assert macs == 58560

# H3 fully connected layer
H3w = init_weights(4*4*12, 4*4*12, 30)
H3b = torch.zeros(30)
assert H3w.numel() + H3b.numel() == 5790
macs += (4*4*12) * 30 + 30
acts += 30
assert macs == 64350

# Output layer
outw = init_weights(30, 30, 10)
outb = -torch.ones(10)
assert outw.numel() + outb.numel() == 310
macs += 30 * 10 + 10
acts += 10
assert macs == 64660

**Task 5** Find out the following features of the model architecture

    * #Parameters: 9760
    * #macs: 64,660
    * #activations:
    * Layer: Conv-Conv-FC-FC

### Score function: Forward pass

The weights, bias, activation function in each layer of neural network define the forward pass. The forward pass is the inference and a major step in training. you'll need to implement TODO section.

In [11]:
def forward(x):
    x = F.pad(x, (2, 2, 2, 2), mode='constant', value=-1.0)
    # TODO H1 layer: Apply convolution with H1w and add H1b
    # Then apply activation and pooling as needed
    x = F.conv2d(x, H1w, stride=2) + H1b
    x = torch.tanh(x)

    # H2 layer
    x = F.pad(x, (2, 2, 2, 2), mode='constant', value=-1.0)
    slice1 = F.conv2d(x[:, 0:8], H2w[0:4], stride=2)
    slice2 = F.conv2d(x[:, 4:12], H2w[4:8], stride=2)
    slice3 = F.conv2d(torch.cat([x[:, 0:4], x[:, 8:12]], dim=1), H2w[8:12], stride=2)
    x = torch.cat((slice1, slice2, slice3), dim=1) + H2b
    x = torch.tanh(x)

    # TODO: H3 fully connected layer
    x = x.flatten(start_dim=1) # (1, 12*4*4)
    x = x @ H3w + H3b
    x = torch.tanh(x)

    #TODO x is now shape (1, 30)
    x = x @ outw + outb
    x = torch.tanh(x)

    #TODO assert final shape (1, 10)
    return x

In [19]:
# Delete the test code later
# Create a dummy image (Batch size 1, Channel 1, 16x16 pixels)
dummy_input = Xte[0:1]

# Run the score function
output_scores = forward(dummy_input)

print(Xte.shape)
print("Output Scores Shape:", output_scores.shape) # Should be [1, 10]
print("Scores per digit:\n", output_scores)

torch.Size([2007, 1, 16, 16])
Output Scores Shape: torch.Size([1, 10])
Scores per digit:
 tensor([[-0.7620, -0.7614, -0.7609, -0.7617, -0.7614, -0.7615, -0.7599, -0.7624,
         -0.7618, -0.7611]], grad_fn=<TanhBackward0>)


### Loss Function: Error

Loss function measures how good the model is in predicting. High loss means the score for wrong digit is high and right digit is low. It is used in ML training for optimizing the network parameters. Error is a more intuitive metric for judgment of overall training performance.

In [21]:
def train_step(optimizer, x, y):
    """Single training step"""
    # Zero gradients
    optimizer.zero_grad()

    # Forward pass and compute loss
    yhat = forward(x)
    loss = torch.mean((y - yhat) ** 2)

    # Backward pass
    loss.backward()

    # Update parameters
    optimizer.step()

    return loss.item()

In [22]:
def eval_split(split, X_tr, Y_tr, X_te, Y_te):
    X, Y = (X_tr, Y_tr) if split == 'train' else (X_te, Y_te)
    yhat = forward(X)

    loss = torch.mean((Y - yhat)**2)
    err = torch.mean((Y.argmax(dim=1) != yhat.argmax(dim=1)).float())
    misses = int(err.item() * Y.shape[0])

    print(f"eval: split {split:5s}. loss {loss.item():e}. error {err.item()*100:.2f}%. misses: {misses}")
    return loss.item(), err.item(), misses

In [24]:
# Parameters
num_epochs = 23
params = [H1w, H1b, H2w, H2b, H3w, H3b, outw, outb]
optimizer = torch.optim.SGD(params, lr=0.03)

for pass_num in range(num_epochs):
    # Perform one epoch of training
    for step_num in range(Xtr.shape[0]):
        # Fetch a single example into a batch of 1
        x, y = Xtr[step_num:step_num+1], Ytr[step_num:step_num+1]

        # Training step
        loss = train_step(optimizer, x, y)

    # After each epoch evaluate the train and test error/metrics
    print(f"\nEpoch {pass_num + 1}/{num_epochs}")
    train_loss, train_err, train_misses = eval_split('train', Xtr, Ytr, Xte, Yte)
    test_loss, test_err, test_misses = eval_split('test', Xtr, Ytr, Xte, Yte)


Epoch 1/23
eval: split train. loss 6.024495e-02. error 8.11%. misses: 590
eval: split test . loss 7.986068e-02. error 11.91%. misses: 239

Epoch 2/23
eval: split train. loss 4.432691e-02. error 5.95%. misses: 433
eval: split test . loss 6.495194e-02. error 10.11%. misses: 203

Epoch 3/23
eval: split train. loss 3.615657e-02. error 5.02%. misses: 366
eval: split test . loss 5.752452e-02. error 8.72%. misses: 174

Epoch 4/23
eval: split train. loss 3.057336e-02. error 4.24%. misses: 309
eval: split test . loss 5.266230e-02. error 7.87%. misses: 158

Epoch 5/23
eval: split train. loss 2.600779e-02. error 3.65%. misses: 266
eval: split test . loss 5.048305e-02. error 7.22%. misses: 144

Epoch 6/23
eval: split train. loss 2.352959e-02. error 3.26%. misses: 237
eval: split test . loss 4.924819e-02. error 7.27%. misses: 145

Epoch 7/23
eval: split train. loss 2.148569e-02. error 2.95%. misses: 215
eval: split test . loss 4.822356e-02. error 7.57%. misses: 152

Epoch 8/23
eval: split train. l

* LeNet-1 Network Class
    * image batch: (7291, 20, 20, 1)→ interpreted as NHWC (batch, height, width, channels) in JAX. (7291, 1, 20, 20) in Pytorch.
    * weights: (12, 1, 5, 5)→ interpreted as (out_channels, in_channels, filter_height, filter_width)
    * In pytorch, NCHW is standard dimension format. you must tell JAX which dimension format you’re using. Correct usage: NHWC input, OIHW kernel. Output dimension same as bias

Key concepts

* Padding
* Convolution
* Matrix Multiplication
* Activation


Epoch 1/23
eval: split train. loss 2.672640e-01. error 49.38%. misses: 3599
eval: split test . loss 2.658389e-01. error 49.58%. misses: 994

Epoch 2/23
eval: split train. loss 6.611320e-02. error 9.70%. misses: 707
eval: split test . loss 6.289694e-02. error 8.97%. misses: 179

Epoch 3/23
eval: split train. loss 4.669868e-02. error 6.83%. misses: 498
eval: split test . loss 4.676422e-02. error 6.98%. misses: 139

Epoch 4/23
eval: split train. loss 3.862600e-02. error 5.47%. misses: 398
eval: split test . loss 4.069002e-02. error 6.03%. misses: 120

Epoch 5/23
eval: split train. loss 3.375371e-02. error 4.94%. misses: 360
eval: split test . loss 3.772741e-02. error 5.73%. misses: 115

Epoch 6/23
eval: split train. loss 3.090480e-02. error 4.53%. misses: 330
eval: split test . loss 3.690541e-02. error 5.58%. misses: 112

Epoch 7/23
eval: split train. loss 2.692368e-02. error 3.96%. misses: 288
eval: split test . loss 3.525820e-02. error 5.43%. misses: 109

Epoch 8/23
eval: split train. 